# 基于决策树的机械故障预测

## 1. 项目背景

随着机械设备的复杂性和自动化程度的提高，传统的维护模式已无法满足日益增长的设备管理需求。  
为确保设备的高效运行并减少突发故障对生产线的影响，越来越多的企业开始依赖预测性维护技术。  
通过提前识别潜在的故障风险，企业能够实施有效的预防性措施，避免设备停机，提升生产效率，并降低维修成本。

本项目旨在建立一个故障预测模型，通过对机械设备和组件的运行数据进行分析，预测哪些设备/组件在未来可能会发生故障。  
我们将综合考虑多种影响因素，如传感器数据（温度、压力、振动等）、历史维护记录、工作环境以及设备运行状态等，利用机器学习算法构建准确的故障预测模型。  
该模型不仅能有效地预测故障发生的时间和类型，还能为维修决策提供科学依据，优化设备的维护计划，提高系统的整体可靠性和生产效率。

## 2. 数据处理

### 2.1 数据概览

该数据描述了多个不同机械设备的特征及其是否存在异常。 

| 数据总数             | 特征总数   |  
|------------------|------------------|
| 20000         | 11            |


特征说明：
- `Unique ID`: 唯一标识符（数据类型：字符串或整数，主要用于区分记录）  
- `Product ID`: 产品标识符（数据类型：分类，用于区分不同产品类型）  
- `Quality`: 产品质量等级（数据类型：分类，可能影响机器状态的特征变量之一）  
- `Ambient T (C)`: 环境温度（数据类型：连续，可能是预测模型的重要因素）  
- `Process T (C)`: 工艺过程温度（数据类型：连续，影响产品质量和机器状态）  
- `Rotation Speed (rpm)`: 旋转速度（数据类型：连续，影响磨损和故障率）  
- `Torque (Nm)`: 扭矩（数据类型：连续，影响机器效率与耐久性）  
- `Tool Wear (min)`: 工具磨损时间（数据类型：连续，提示需要维护和更换工具）  
- `Machine Status`: 机器状态（目标变量，数据类型：分类，0（正常）或 1（故障））

首先我们导入相应数据库并读取数据：

In [ ]:
import pandas as pd  # 导入pandas库并简写为pd，pandas是一个数据处理库，提供了大量的数据处理函数，可以用来处理数据。
import numpy as np  # 导入numpy库并简写为np，numpy是一个数值计算库，提供了大量的数值计算函数，可以用来进行数值计算。

In [ ]:
df = pd.read_csv('/home/jovyan/work/datasets/679333b010f300038decaf07-momodel/factory_data_classification.csv')  # 读取数据

In [ ]:
#展示前5行数据
df.head() 

### 2.2 删除无用特征

- 删除唯一标识符： `Unique ID` 列，因为这些特征与模型及组件故障预测不相关
- 删除存在特征值为 0 的数据，为了后续更好的计算

In [ ]:
df = df.drop(['Unique ID'],axis=1)  # 删除数据中的'Unique ID'列。

In [ ]:
columns_with_zero = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] and 0 in df[col].values]
columns_with_zero

注意，这里不能删除`Machine Status`列中为 0 的值，因为这是表示是否损坏。

In [ ]:
df = df.loc[(df[['Tool Wear (min)']] != 0).all(axis=1)]  # 删除数据中列中存在0值的行。

### 2.3 填充数据

首先查看是否有为空值的列，如果有，需要进行填充。

In [ ]:
display(df.isnull().sum())  # 显示数据集中的缺失值数量

我们需要针对不同数据类型采用不同填充方法，下面我们选择对`Quality`列进行众数填充，对`Process T (C)`，`Rotation Speed (rpm)`进行平均值填充。

- 使用众数填充类别特征的缺失值
- 连续型特征使用不同方法填充：
  - 平均值填充
  - 中位数填充
  - KNN 填充

In [ ]:
mode_quality = df['Quality'].mode()[0]
print(mode_quality)

In [ ]:
# 使用众数填充类别型特征的缺失值
df['Quality'].fillna(mode_quality, inplace=True)

In [ ]:
# 使用平均值填充连续型特征的缺失值
df['Process T (C)'] = df['Process T (C)'].fillna(df['Process T (C)'].mean())
df['Rotation Speed (rpm)'] = df['Rotation Speed (rpm)'].fillna(df['Rotation Speed (rpm)'].mean())

填充完我们再运行一次，可以发现已经没有缺失值了：

In [ ]:
display(df.isnull().sum())

### 2.4 特征工程

- 创建新特征提高模型预测准确度，增强数据相关性。  
- 这是一个将人工特征设计到算法中的过程，从而提高其性能。

首先将部分列名改成和业务符合逻辑的名字，便于后续处理。

In [ ]:
df.rename(columns = {'Tool Wear (min)':'Tool Lifespan (min)'}, inplace = True)  # 将数据中的'Tool Wear (min)'列重命名为'Tool Lifespan (min)'
df['Product ID'] = df['Product ID'].astype(str).str[0]  # 将数据中的'Product ID'列的数据转换为字符串，并取每个字符串的第一个字符，因为这里只需要前面的字符来区分
df = df.rename(columns={'Product ID':'Product Code'})  # 将数据中的'Product ID'列重命名为'Product Code'

为了更好的做预测，我们新增了三个特征：  
- `T Difference Squared (C^2)`温度差平方可以突出温度变化的较大差异，可以研究机器过程中的温度差平方与其可靠性之间的关系。
- `Tool Lifespan/Temp Increase^2 (min/C^2)` 工具寿命与温差平方的比值，可以研究温度增加对工具寿命的影响。
- `Horsepower (HP)` 马力是制造商需要考虑的一个重要条件，因为它直接反映了机器的性能，可以研究机器的马力与其可靠性之间的关系。

In [ ]:
df.loc[:, ['T Difference Squared (C^2)']] = (df['Ambient T (C)']- df['Process T (C)'])**2  # 计算'Ambient T (C)'和'Process T (C)'列的差的平方，并将结果存储在'T Difference Squared (C^2)'列。
df.loc[:, ['Tool Lifespan/Temp Increase^2 (min/C^2)']] = round(df['Tool Lifespan (min)'] /df['T Difference Squared (C^2)'])  # 计算'Tool Lifespan (min)'和'T Difference Squared (C^2)'列的商，并将结果存储在'Tool Lifespan/Temp Increase^2 (min/C^2)'列。
df.loc[:, ['Horsepower (HP)']] = (df['Rotation Speed (rpm)'] * df['Torque (Nm)']) /5252  # 计算'Rotation Speed (rpm)'和'Torque (Nm)'列的乘积除以5252，并将结果存储在'Horsepower (HP)'列。

我们再看看现在的数据样式：

In [ ]:
df.head()

In [ ]:
df.describe()

## 3. 探索性数据分析

### 3.1 类别型数据样本特征分布

我们需要统计`Product Code`，`Quality`两列的分布情况：

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

pie_arr_product = Counter(df['Product Code'])  # 统计数据集中'Product Code'列中各个元素的数量。

plt.figure(figsize=(25,18),dpi=52)  # 创建一个大小为25*18的图表。
plt.subplot(1,2,1)  # 创建一个子图，1行2列，第1列。
# 绘制饼图，第一个参数为数据，第二个参数为标签，textprops表示文本属性
plt.pie(pie_arr_product.values(), labels=pie_arr_product.keys(), autopct='%1.1f%%', startangle=140,textprops={'fontsize': 34})  
plt.title('Product Code',fontsize=50)  # 设置标题，字体大小为50。

pie_arr_quality = Counter(df['Quality'])  # 统计数据集中'Quality'列中各个元素的数量。

plt.subplot(1,2,2)
plt.pie(pie_arr_quality.values(), labels=pie_arr_quality.keys(), autopct='%1.1f%%', startangle=140,textprops={'fontsize': 34})
plt.title('Quality of Product',fontsize=50)
plt.suptitle('数据样本类别型特征分布',weight='bold',fontsize=60)
plt.show()

如上图，我们可以知道产品代码的分布大致均匀地分为四类，这表明产品代码可能对机器状态没有显著影响。

然而，我们可以观察到，产品质量的分布中，大约 62% 的产品是低质量，29% 的产品是中等质量，只有 9% 的产品是高质量。所以，产品的质量可能会对机器的最终状态产生影响。

### 3.2 可视化 Horsepower 曲线

由于 Horsepower = $\frac{{\sf RPM} \times {\sf Torque}}{5252} $    

我们可以使用机器的马力值（Horsepower）来衡量其原始性能，并研究其作为工程特征的有效性。

In [ ]:
import plotly.express as px
# 使用plotly.express绘制散点图
fig = px.scatter(df, x='Torque (Nm)', y='Rotation Speed (rpm)',
                 marginal_x="histogram", color='Rotation Speed (rpm)',
                 marginal_y="histogram", trendline_color_override='grey')
# marginal_x和marginal_y表示x轴和y轴的直方图，color表示颜色，trendline表示趋势线，trendline_color_override表示趋势线颜色

# 设置图表的高度和宽度
fig.update_layout(height=700, width=1000)

# 设置图表标题、y轴标题和x轴标题
fig.update_layout(
    title='Horsepower 曲线',
    yaxis_title="Torque (Nm)",
    xaxis_title="Rotation Speed (rpm)"
)

# 显示图表
fig.show()

我们可以看到，旋转速度的分布范围明显大于扭矩的分布范围。  
这表明，扭矩值更集中在均值附近，暗示扭矩值更可靠，相较于旋转速度，扭矩更具一致性。

### 3.3 温差对工具寿命影响的可视化

In [ ]:
# # 首先需要安装如下库，如果已有，则不需要运行。
# pip install statsmodels

In [ ]:
pip install statsmodels

In [ ]:
import pandas as pd
import plotly.express as px
from statistics import mean
# 从df中提取'T Difference Squared (C^2)'和'Tool Lifespan (min)'列，并复制。
plot_df = df.copy()

# 使用plotly.express绘制散点图
fig = px.scatter(plot_df, 
                 x='T Difference Squared (C^2)', 
                 y='Tool Lifespan (min)', 
                 marginal_x="histogram", 
                 marginal_y="histogram", 
                 color='Tool Lifespan (min)', 
                 trendline="ols", 
                 trendline_color_override="red")

# 更新图表的模式为markers
fig.data[0].update(mode='markers')

# 设置图表标题、y轴标题和x轴标题
fig.update_layout(
    title='温差与工具寿命的关系图',
    yaxis_title="Lifespan (min)",
    xaxis_title="Temperature Increase (C^2)"
)

# 显示图表
fig.show()

# 计算'T Difference Squared (C^2)'和'Tool Lifespan (min)'列的相关系数，method='pearson'表示使用皮尔逊相关系数
# correlation = plot_df.corr(method='pearson')
# display(correlation)

正如我们所看到的，相关性矩阵显示工具寿命与温差之间几乎没有相关性，相关系数为 0.65%。  
由于相关系数仅为 0.65%，我们可以放心地认为这两个值之间的相关性不足以对整体机器状态产生强烈的影响。

## 4. 机器学习模型探索


### 4.1 划分训练集与测试集

在故障检测场景下，当正类预测非常少（甚至没有）时，F1 分数仍然是一个稍微更好的指标。  
这是因为机器被认为故障并修复总比机器被认为正常但实际上故障更好，因为后者可能导致更多的停机时间，直到识别出正确的机器并修复它。

- 使用 pandas 的 get_dummies 函数将 Quality 和 Product Code 转换为独热编码（one-hot encoding）
- 将 Machine Status 列转换为数值标签

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

label_encoder_quality = LabelEncoder()
label_encoder_product_code = LabelEncoder()

df['Quality'] = label_encoder_quality.fit_transform(df['Quality'])
df['Product Code'] = label_encoder_product_code.fit_transform(df['Product Code'])

# 将 Machine Status 转换为数值标签
label_encoder_machine_status = LabelEncoder()
df['Machine Status'] = label_encoder_machine_status.fit_transform(df['Machine Status'])

选择特征和目标变量：

In [ ]:
X = df.drop('Machine Status', axis=1)
y = df['Machine Status']

划分训练集和测试集：

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

### 4.2 使用决策树预测

In [ ]:
# 创建决策树分类器实例
clf = DecisionTreeClassifier(random_state=4)

# 训练模型
clf.fit(X_train, y_train)

In [ ]:
# 在测试集上进行预测
y_pred = clf.predict(X_test)

In [ ]:
# 计算准确率
accuracy = np.mean(y_pred == y_test)

print(f"Accuracy: {accuracy:.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# 计算混淆矩阵
cm = confusion_matrix(y_test, y_pred)

# 打印分类报告
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 可视化混淆矩阵
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score, ShuffleSplit

# 交叉验证
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean accuracy: {cv_scores.mean():.2f}")


可以看到决策树在检测正常机器和故障机器的真正正例（True Positive）方面表现较好，尤其是在处理复杂的非线性数据时。    

本次试验的准确率为 99%，可能的原因：


1.   本次数据样本不平衡，样本中机器故障和机器正常的数量不平衡，可以尝试数据增强
2.   采用填充空值时选择的均值和众数的简单方法，可以尝试其他的填充方法，如 KNN 填充等



## 5. 总结

Q.1: 预测任务是如何定义的？  
A.1: 使用给定的特征，模型在多大程度上能够准确预测机器/组件在实际故障前可能发生故障的情况。

Q.2: 模型输出变量的含义是什么？  
A.2: 表示机器的状态；即机器是否发生故障，1表示故障，0表示正常。

Q.3: 如何将数据表示为特征？  
A.3: 在机器学习中，特征是单个可测量的属性。在本例中，可以将列变量表示为模型中的特征，如 `Ambient T (C)`和 `Rotation Speed (rpm)`。

Q.4: 如何进行特征处理？    
A.4: 添加了三个新的特征:  
- `T Difference Squared (C^2)`温度差平方可以突出温度变化的较大差异，这将为模型提供更具影响力的数据。我们可以研究机器过程中的温度差平方与其可靠性之间的关系。  
- `Tool Lifespan/Temp Increase^2 (min/C^2)` 工具寿命与温差平方的比值，研究温度增加对工具寿命的影响。  
- `Horsepower (HP)` 马力是制造商需要考虑的一个重要条件，因为它直接反映了机器的性能。可以研究机器的马力与其可靠性之间的关系。    

Q.5: 如何进行数据处理？   
A.5: 查看是否有空值和为 0 的值、然后进行补全数据。

Q.6: 决策树的优缺点？
A.6:
             
    决策树模型具有分裂数据的能力，它能够适应不同类型的数据，尤其是非线性特征。这使得决策树相比于逻辑回归更能有效地捕捉数据中的复杂关系。    

    决策树也存在一些局限性。例如，决策树对数据中的噪声和异常值非常敏感。少量的噪声数据可能会导致树的结构变化，进而影响模型的表现。此外，决策树容易发生过拟合，尤其是在树的深度没有适当限制时。 

    对于缺失值，决策树能够处理这些问题，通过数据划分时忽略缺失的样本。缺失值处理方法仍然是一个重要的因素，若没有适当的预处理，可能会影响模型的表现。



参考资料  
- https://datascience.stackexchange.com/questions/90175/comparing-ml-models-to-baselines
- https://www.jstor.org/stable/44547452
- Schneider, E.W., Blossfeld, D.H. and Balnaves, M.A. (1988). Effect of Speed and Power Output on Piston Ring Wear in a Diesel Engine. SAE Transactions, [online] 97, pp.1257–1267. Available at: https://www.jstor.org/stable/44547452 [Accessed 9 Jun. 2022].